In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Проверка GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используем устройство: {device}")

# Используем зеркало
torchvision.datasets.CIFAR10.url = "https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz"

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

Используем устройство: cuda


100%|██████████| 170M/170M [00:11<00:00, 14.5MB/s]


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import time

def hinton_distillation_loss(student_logits, teacher_logits, true_labels, T=4.0, alpha=0.9):
    hard_loss = F.cross_entropy(student_logits, true_labels)
    soft_student = F.log_softmax(student_logits / T, dim=1)
    soft_teacher = F.softmax(teacher_logits / T, dim=1)
    soft_loss = F.kl_div(soft_student, soft_teacher, reduction='batchmean') * (T * T)
    return (1.0 - alpha) * hard_loss + alpha * soft_loss

# Загружаем Учителя (ResNet-56, 0.85M параметров, ~94% acc)
print("Загружаем Учителя...")
teacher_model = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_resnet56", pretrained=True).to(device)
teacher_model.eval()

# Функция для создания чистого Ученика (ResNet-20, 0.27M параметров)
def get_fresh_student():
    return torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_resnet20", pretrained=False).to(device)

Загружаем Учителя...
The repository chenyaofo_pytorch-cifar-models does not belong to the list of trusted repositories and as such cannot be downloaded. Do you trust this repository and wish to add it to the trusted list of repositories (y/N)?y
Downloading: "https://github.com/chenyaofo/pytorch-cifar-models/zipball/master" to /root/.cache/torch/hub/master.zip
Downloading: "https://github.com/chenyaofo/pytorch-cifar-models/releases/download/resnet/cifar10_resnet56-187c023a.pt" to /root/.cache/torch/hub/checkpoints/cifar10_resnet56-187c023a.pt


100%|██████████| 3.39M/3.39M [00:00<00:00, 60.4MB/s]


In [ ]:
print("--- ЭТАП 1: Обучаем Ученика с нуля (Baseline) ---")
student_base = get_fresh_student()
optimizer_base = optim.Adam(student_base.parameters(), lr=0.001)
scheduler_base = CosineAnnealingLR(optimizer_base, T_max=20)

epochs = 20

for epoch in range(epochs):
    running_loss = 0.0
    start_time = time.time()
    student_base.train()
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer_base.zero_grad()
        loss = F.cross_entropy(student_base(inputs), labels)
        loss.backward()
        optimizer_base.step()
        running_loss += loss.item()


    student_base.eval() # Переводим в режим инференса
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = student_base(inputs)

            # Берем индекс максимального значения как предсказанный класс
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()


    scheduler_base.step()
    val_accuracy = 100 * correct / total
    epoch_time = time.time() - start_time
    print(f"Эпоха [{epoch+1}/{epochs}] | Loss: {running_loss/len(train_loader):.4f} | Точность на тесте: {val_accuracy:.2f}% | Время: {epoch_time:.1f} сек")



student_base.eval()
correct, total = 0, 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        _, predicted = torch.max(student_base(inputs), 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
base_acc = 100 * correct / total
print(f"ИТОГ Baseline: {base_acc:.2f}%")

--- ЭТАП 1: Обучаем Ученика с нуля (Baseline) ---


Using cache found in /root/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


Эпоха [1/20] | Loss: 1.3915 | Точность на тесте: 51.22% | Время: 20.3 сек
Эпоха [2/20] | Loss: 0.9979 | Точность на тесте: 63.87% | Время: 20.4 сек
Эпоха [3/20] | Loss: 0.8264 | Точность на тесте: 69.87% | Время: 20.1 сек
Эпоха [4/20] | Loss: 0.7017 | Точность на тесте: 63.73% | Время: 20.6 сек
Эпоха [5/20] | Loss: 0.6076 | Точность на тесте: 71.18% | Время: 20.0 сек
Эпоха [6/20] | Loss: 0.5304 | Точность на тесте: 75.70% | Время: 20.8 сек
Эпоха [7/20] | Loss: 0.4605 | Точность на тесте: 73.53% | Время: 21.2 сек
Эпоха [8/20] | Loss: 0.3975 | Точность на тесте: 77.93% | Время: 20.1 сек
Эпоха [9/20] | Loss: 0.3365 | Точность на тесте: 76.82% | Время: 20.9 сек
Эпоха [10/20] | Loss: 0.2715 | Точность на тесте: 75.17% | Время: 20.1 сек
Эпоха [11/20] | Loss: 0.2168 | Точность на тесте: 78.38% | Время: 20.8 сек
Эпоха [12/20] | Loss: 0.1617 | Точность на тесте: 77.35% | Время: 20.7 сек
Эпоха [13/20] | Loss: 0.1166 | Точность на тесте: 78.12% | Время: 20.3 сек
Эпоха [14/20] | Loss: 0.0817 | Точ

In [ ]:
print("\n--- ЭТАП 2: Обучаем Ученика через Дистилляцию (Knowledge Distillation) ---")
student_kd = get_fresh_student()
optimizer_kd = optim.Adam(student_kd.parameters(), lr=0.001)
scheduler_kd = CosineAnnealingLR(optimizer_kd, T_max=20)

epochs = 20

for epoch in range(epochs):
    running_loss = 0.0
    start_time = time.time()

    student_kd.train()
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer_kd.zero_grad()

        student_logits = student_kd(inputs)
        with torch.no_grad():
            teacher_logits = teacher_model(inputs)

        loss = hinton_distillation_loss(student_logits, teacher_logits, labels, T=4.0, alpha=0.9)
        loss.backward()
        optimizer_kd.step()
        running_loss += loss.item()


    student_kd.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = student_kd(inputs)

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    scheduler_kd.step()
    val_accuracy = 100 * correct / total
    epoch_time = time.time() - start_time
    print(f"Эпоха [{epoch+1}/{epochs}] | Loss: {running_loss/len(train_loader):.4f} | Точность на тесте: {val_accuracy:.2f}% | Время: {epoch_time:.1f} сек")


student_kd.eval()
correct, total = 0, 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        _, predicted = torch.max(student_kd(inputs), 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
kd_acc = 100 * correct / total

teacher_model.eval()
correct, total = 0, 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        _, predicted = torch.max(teacher_model(inputs), 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
teacher_acc = 100 * correct / total

print(f"\n===== РЕЗУЛЬТАТЫ =====")
print(f"Учитель (ResNet-56): {teacher_acc}%")
print(f"Ученик сам по себе (ResNet-20) [Бейзлайн]: {base_acc:.2f}%")
print(f"Ученик с Дистилляцией: {kd_acc:.2f}%")
print(f"ПРИРОСТ ОТ ДИСТИЛЛЯЦИИ: +{(kd_acc - base_acc):.2f}%")


--- ЭТАП 2: Обучаем Ученика через Дистилляцию (Knowledge Distillation) ---


Using cache found in /root/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


Эпоха [1/20] | Loss: 9.2033 | Точность на тесте: 54.65% | Время: 27.5 сек
Эпоха [2/20] | Loss: 6.4077 | Точность на тесте: 66.41% | Время: 28.2 сек
Эпоха [3/20] | Loss: 5.2485 | Точность на тесте: 70.03% | Время: 26.1 сек
Эпоха [4/20] | Loss: 4.4088 | Точность на тесте: 70.59% | Время: 27.8 сек
Эпоха [5/20] | Loss: 3.8263 | Точность на тесте: 73.68% | Время: 26.6 сек
Эпоха [6/20] | Loss: 3.4163 | Точность на тесте: 73.53% | Время: 26.3 сек
Эпоха [7/20] | Loss: 3.0374 | Точность на тесте: 75.84% | Время: 26.2 сек
Эпоха [8/20] | Loss: 2.6997 | Точность на тесте: 77.12% | Время: 26.5 сек
Эпоха [9/20] | Loss: 2.3658 | Точность на тесте: 76.29% | Время: 26.2 сек
Эпоха [10/20] | Loss: 2.0642 | Точность на тесте: 76.22% | Время: 26.1 сек
Эпоха [11/20] | Loss: 1.7954 | Точность на тесте: 79.35% | Время: 26.4 сек
Эпоха [12/20] | Loss: 1.5368 | Точность на тесте: 78.97% | Время: 26.5 сек
Эпоха [13/20] | Loss: 1.3135 | Точность на тесте: 80.47% | Время: 26.3 сек
Эпоха [14/20] | Loss: 1.1338 | Точ